In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import seaborn as snspip 
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from utils import get_hpi

In [2]:
data = pd.read_csv('../data/merged.csv')

postcode_encoder = LabelEncoder()
data['Postcode_encoded'] = postcode_encoder.fit_transform(data['Postcode'].astype(str))

city_encoder = LabelEncoder()
data['City_encoded'] = city_encoder.fit_transform(data['City'].astype(str))

data['SoldDate'] = data['SoldDate'].str.replace('/span', '', regex=False)
data['SoldDate'] = pd.to_datetime(data['SoldDate'], format='%d-%b-%y', errors='coerce')
data.dropna(subset=['SoldDate'], inplace=True)
data['SoldYear'] = data['SoldDate'].dt.year

invalid_dates = data[data['SoldDate'].isna()]
print(invalid_dates[['SoldDate', 'Price', 'Postcode', 'City']])

data['Price'] = pd.to_numeric(data['Price'], errors='coerce')

# Drop rows where Price could not be converted (i.e. NaN)
data.dropna(subset=['Price'], inplace=True)

Empty DataFrame
Columns: [SoldDate, Price, Postcode, City]
Index: []


In [3]:
data['SaleHPI'] = data['SoldYear'].apply(get_hpi)
data['CurrentHPI'] = 1.00  # 2024 is our baseline
data['InflationFactor'] = data['CurrentHPI'] / data['SaleHPI']

# 6. Create a new column for the adjusted price
data['Price_Adjusted2024'] = data['Price'] * data['InflationFactor']

# 7. Now define X and y for modeling
#    (Here, you could choose 'Price_Adjusted2024' or 'Price' as your target, 
#     depending on whether you want to train on inflation-adjusted values.)

y = data['Price_Adjusted2024'].values
X = data[['City_encoded']] 

print(f"Number of rows in X: {X.shape[0]}")
print(f"Number of rows in y: {y.shape[0]}")

Number of rows in X: 52789
Number of rows in y: 52789


In [4]:
from xgboost import XGBRegressor
from xgboost.callback import EarlyStopping
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42)

y_train_log = np.log1p(y_train) 
y_test_log = np.log1p(y_test)

callbacks = [
    EarlyStopping(
        rounds=10,
        save_best=True,
        maximize=False,  
        data_name='validation_0',
        metric_name='rmse' 
    )
]


xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    early_stopping_rounds=10,
    callbacks=callbacks
)


# Train

In [5]:
xgb_model.fit(
    X_train,
    y_train_log,
    eval_set=[(X_test, y_test_log)],
    verbose=True,

)
y_pred_log = xgb_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

[0]	validation_0-rmse:0.56557
[1]	validation_0-rmse:0.56311
[2]	validation_0-rmse:0.56087
[3]	validation_0-rmse:0.55884
[4]	validation_0-rmse:0.55700
[5]	validation_0-rmse:0.55532
[6]	validation_0-rmse:0.55381
[7]	validation_0-rmse:0.55242
[8]	validation_0-rmse:0.55117
[9]	validation_0-rmse:0.55003
[10]	validation_0-rmse:0.54901
[11]	validation_0-rmse:0.54808
[12]	validation_0-rmse:0.54716
[13]	validation_0-rmse:0.54631
[14]	validation_0-rmse:0.54554
[15]	validation_0-rmse:0.54483
[16]	validation_0-rmse:0.54396
[17]	validation_0-rmse:0.54317
[18]	validation_0-rmse:0.54247
[19]	validation_0-rmse:0.54164
[20]	validation_0-rmse:0.54105
[21]	validation_0-rmse:0.54032
[22]	validation_0-rmse:0.53984
[23]	validation_0-rmse:0.53936
[24]	validation_0-rmse:0.53871
[25]	validation_0-rmse:0.53827
[26]	validation_0-rmse:0.53770
[27]	validation_0-rmse:0.53747
[28]	validation_0-rmse:0.53724
[29]	validation_0-rmse:0.53704
[30]	validation_0-rmse:0.53672
[31]	validation_0-rmse:0.53642
[32]	validation_0-

# Evaluate

In [18]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print("\nEvaluation metrics (City-Only Model):")
print(f"MAE:  {mae:,.2f}")
print(f"MSE:  {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")


Evaluation metrics (City-Only Model):
MAE:  339,150.55
MSE:  276,568,047,197.60
RMSE: 525,897.37
MAE:  892,266.31
MSE:  1,110,303,599,302.23
RMSE: 1,053,709.45


# Prediction check

In [16]:
# sample_postcode = 5067
sample_city = "Kent Town"
# encoded_postcode = postcode_encoder.transform([str(sample_postcode)])[0]
encoded_city = city_encoder.transform([str(sample_city)])[0]
sample_input = np.array([[ encoded_city]], dtype='float32')
single_prediction_log = xgb_model.predict(sample_input)[0]
single_prediction_price = np.expm1(single_prediction_log)
print(f"\nSingle prediction  City={sample_city}: {single_prediction_price:,.2f}")


Single prediction  City=Kent Town: 912,033.00
